In [7]:
from pathlib import Path

import pandas as pd
from datetime import date

from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import LETTER
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Image, Table, TableStyle, KeepTogether
)
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib import colors

from PIL import Image as PILImage

In [8]:
class NumberedCanvas(canvas.Canvas):
    """
    Canvas that allows 'Page X of Y' by saving page states and rendering
    footer content in a second pass without duplicating pages.
    """
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._saved_page_states = []

    def showPage(self):
        # Save the state of each page as it is created
        self._saved_page_states.append(dict(self.__dict__))
        # Start a new page (do NOT call super().showPage() here)
        self._startPage()

    def save(self):
        page_count = len(self._saved_page_states)
        for state in self._saved_page_states:
            self.__dict__.update(state)
            self.draw_footer(page_count)
            # Finalize this page and move to next without creating duplicates
            canvas.Canvas.showPage(self)
        canvas.Canvas.save(self)

    def draw_footer(self, page_count: int):
        width, height = LETTER

        y = 0.55 * inch
        left_x   = 0.75 * inch
        center_x = width / 2.0
        right_x  = width - 0.75 * inch

        self.setFont("Helvetica-Oblique", 9)
        self.setFillColor(colors.grey)

        # Left: fixed text
        self.drawString(left_x, y, "From the offices of Boondoggle Research")

        # Center: date
        self.drawCentredString(center_x, y, date.today().isoformat())

        # Right: page x of y
        self.drawRightString(right_x, y, f"Page {self._pageNumber} of {page_count}")


In [9]:
def build_pdf_report(
    df: pd.DataFrame,
    out_pdf: str = "Kilbreths_Pig_Report.pdf",
    report_title_left: str = "KPI Report",
    footer_text: str = "From the offices of Boondoggle Research",
    evaluated_article: str = "",
    about_sentence: str = "This report summarizes automated citation verification results for the reference list.",
    logo_path: str = "logo.png",
    yes_icon: str = "yes.png",
    maybe_icon: str = "maybe.png",
    no_icon: str = "no.png",
    confidence_col: str = "confidence",
    refnum_col: str = "ref_num",
    title_col: str = "title",
    url_col: str = "url",
):
    """
    Build a PDF report from a reference dataframe.

    Rules:
    - References are ordered by ref_num.
    - Do NOT use the dataframe index in the output.
    - First bit of info is ref number.
    - Then paper title; if missing, use URL.
    - Inline icon based on confidence: high->yes.png, maybe->maybe.png, low->no.png
    - Header: left title text; right logo.
    - Footer: centered 'From the offices of Boondoggle Research'
    """

    # --- paths / validation ---
    out_pdf = str(out_pdf)
    logo_path = Path(logo_path)
    yes_icon = Path(yes_icon)
    maybe_icon = Path(maybe_icon)
    no_icon = Path(no_icon)

    for p in [yes_icon, maybe_icon, no_icon]:
        if not p.exists():
            raise FileNotFoundError(f"Missing required icon file: {p}")

    if logo_path.exists() is False:
        # Not fatal; you said you'll provide it. We'll just omit if missing.
        logo_path = None

    # --- sort / clean ---
    required = [refnum_col, confidence_col, url_col]
    for c in required:
        if c not in df.columns:
            raise KeyError(f"DataFrame missing required column: '{c}'")

    if title_col not in df.columns:
        # If you truly don't have titles at all, we'll treat as always missing and use URL.
        df = df.copy()
        df[title_col] = None

    d = df.copy()
    d = d.sort_values(refnum_col, kind="mergesort")  # stable sort

    # --- styles ---
    styles = getSampleStyleSheet()
    body = ParagraphStyle(
        "body",
        parent=styles["BodyText"],
        fontName="Helvetica",
        fontSize=10,
        leading=13,
        spaceAfter=2,
    )
    refnum_style = ParagraphStyle(
        "refnum",
        parent=body,
        fontName="Helvetica-Bold",
    )
    small = ParagraphStyle(
        "small",
        parent=body,
        fontSize=9,
        leading=11,
        textColor=colors.grey,
    )

    yes_style = ParagraphStyle(
        "yes_style",
        parent=body,
        textColor=colors.darkgreen,
    )

    maybe_style = ParagraphStyle(
        "maybe_style",
        parent=body,
        textColor=colors.black,
    )

    no_style = ParagraphStyle(
        "no_style",
        parent=body,
        textColor=colors.red,
    )

    warn_style = ParagraphStyle(
        "warn_style",
        parent=body,
        fontSize=9,
        leading=11,
        textColor=colors.red,
    )


    def draw_header(canvas, doc):
        canvas.saveState()

        width, height = LETTER

        # Header baseline
        y_top = height - 0.75 * inch

        # Left header text
        canvas.setFont("Helvetica-Bold", 12)
        canvas.drawString(0.75 * inch, y_top, report_title_left)

        # Right logo (aspect-preserving)
        if logo_path is not None:
            try:
                with PILImage.open(logo_path) as im:
                    w_px, h_px = im.size
                    aspect = h_px / w_px

                logo_w = 1.8 * inch
                logo_h = logo_w * aspect

                # Nudge up/right without moving anything else
                x_logo = width - 0.75 * inch - logo_w + 0.25 * inch
                y_logo = height - 0.75 * inch - logo_h + 0.45 * inch

                canvas.drawImage(
                    str(logo_path),
                    x_logo, y_logo,
                    width=logo_w,
                    height=logo_h,
                    preserveAspectRatio=True,
                    mask="auto",
                )
            except Exception:
                pass

        canvas.restoreState()

    
    def pick_text_style(conf):
        c = (conf or "").lower().strip()
        if c in {"high", "yes", "verified"}:
            return yes_style
        if c in {"maybe", "medium", "uncertain"}:
            return maybe_style
        return no_style


        canvas.setFont("Helvetica-Oblique", 9)
        canvas.setFillColor(colors.grey)
        canvas.drawString(0.75 * inch, 0.55 * inch, footer_text)

        canvas.restoreState()

    def _norm_status(s):
        return (s or "").strip().lower()

    def _is_blocked(row) -> bool:
        status = (row.get(confidence_col) or "").strip().lower()
        notes  = (row.get("verify_notes") or "").strip().lower()
        best   = (row.get("verify_best_source") or "").strip().lower()
        evid   = (row.get(url_col) or "").strip().lower()  # usually verify_evidence_url

        # explicit blocked statuses
        if status in {"blocked", "host_blocked", "site_blocked", "blocked_by_host"}:
            return True

        # common textual signals (notes or best-source)
        blocked_signals = (
            "blocked", "access denied", "forbidden", "403", "captcha",
            "cloudflare", "rate limit", "robot", "robots", "paywall",
            "denied", "unauthorized", "429", "too many requests"
        )
        if any(s in notes for s in blocked_signals):
            return True
        if any(s in best for s in blocked_signals):
            return True

        # **important**: verification says "maybe" but we have no evidence URL
        # This is effectively "blocked or unresolvable automatically" for your workflow.
        if status in {"maybe", "uncertain", "partial"} and evid == "":
            return True

        return False

    def _best_url(row) -> str:
        # Prefer verified evidence URL, else LLM URL, else raw urls field
        for col in [url_col, "url_llm", "urls"]:
            u = (row.get(col) or "").strip()
            if u:
                return u
        return ""

    def _fmt_ref_list(ref_ids):
        ref_ids = [str(int(x)) for x in ref_ids if pd.notna(x)]
        if not ref_ids:
            return ""
        if len(ref_ids) <= 12:
            return ", ".join(ref_ids)
        # If long, show first few and last few (keeps the sentence readable)
        return ", ".join(ref_ids[:10]) + ", …, " + ", ".join(ref_ids[-3:])


    
    # --- document ---
    doc = SimpleDocTemplate(
        out_pdf,
        pagesize=LETTER,
        leftMargin=0.75 * inch,
        rightMargin=0.75 * inch,
        topMargin=1.25 * inch,
        bottomMargin=0.85 * inch,
        title=report_title_left,
        author="Kilbreth's Pig",
    )

    story = []

    # Optional: small spacer under header area
    story.append(Spacer(1, 0.2 * inch))

    # --- lead-in paragraphs (auto-generated) ---
    # Categorize rows
    d2 = d.copy()

    confirmed_mask = d2[confidence_col].apply(_norm_status).isin({"high", "yes", "verified"})
    low_mask       = d2[confidence_col].apply(_norm_status).isin({"low", "no", "missing", "bad", "failed"})
    blocked_mask   = d2.apply(_is_blocked, axis=1)

    n_total = len(d2)
    n_confirmed = int(confirmed_mask.sum())
    n_blocked = int(blocked_mask.sum())
    n_bad = int((low_mask & ~blocked_mask).sum())   # treat blocked separately

    blocked_refs = d2.loc[blocked_mask, refnum_col].tolist()
    blocked_refs_str = _fmt_ref_list(blocked_refs)


    lead_style = ParagraphStyle(
        "lead_style",
        parent=body,
        fontSize=10,
        leading=14,
        spaceAfter=6,
    )

    lead_bold = ParagraphStyle(
        "lead_bold",
        parent=lead_style,
        fontName="Helvetica-Bold",
    )

    # (1) What the report is about (<= 1 sentence)
    story.append(Paragraph(about_sentence, lead_style))

    # (2) Article under evaluation
    if evaluated_article.strip():
        story.append(Paragraph(f"Article under evaluation: <b>{evaluated_article}</b>.", lead_style))

    # (3) Brief synopsis of results
    story.append(Paragraph(
        f"Synopsis: {n_confirmed} confirmed good, {n_blocked} blocked by host site, {n_bad} likely bad references (of {n_total} total).",
        lead_style
    ))

    # (4) Instruction for blocked sites
    if n_blocked > 0:
        story.append(Paragraph(
            f"Blocked sites: for reference number(s) <b>{blocked_refs_str}</b>, click the listed URL to manually verify the citation.",
            lead_style
        ))

    story.append(Spacer(1, 0.20 * inch))


    # --- build entries ---
    icon_size = 0.32 * 1.1 * inch  # small inline icon
    col_icon_w = 0.45 * 1.1 * inch
    col_text_w = doc.width - col_icon_w

    def pick_icon(conf: str) -> Path:
        c = (conf or "").strip().lower()
        if c in {"high", "yes", "y", "true", "exists"}:
            return yes_icon
        if c in {"maybe", "medium", "uncertain", "possible"}:
            return maybe_icon
        # low / no / unknown
        return no_icon

    for _, row in d.iterrows():
        ref_num = row.get(refnum_col)
        conf = row.get(confidence_col)
        title = row.get(title_col)
        url = row.get(url_col)

        title = (title or "").strip()
        url = (url or "").strip()

        best_url = _best_url(row)

        display_text = title if title else best_url
        if not display_text:
            display_text = "(missing title and url)"

        # If blocked, force URL display (so summary + instructions stay true)
        if _is_blocked(row):
            display_text = best_url if best_url else display_text


    # If blocked, force URL display (so the instruction makes sense)
        if _is_blocked(row):
            display_text = url if url else display_text


        # Row content: icon + "N. Title/URL"
        icon_path = pick_icon(conf)
        icon = Image(str(icon_path), width=icon_size, height=icon_size)

        refnum_para = Paragraph(f"{int(ref_num)}.", refnum_style) if pd.notna(ref_num) else Paragraph("?.", refnum_style)
        text_style = pick_text_style(conf)
        main_para = Paragraph(display_text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;"),
            text_style
        )


        # Put refnum + text together (so ref number is first “bit of information”)
        # Layout: [icon] [refnum + main text]
        text_table = Table(
            [[refnum_para, main_para]],
            colWidths=[0.35 * inch, col_text_w - 0.35 * inch],
            style=TableStyle([
                ("VALIGN", (0, 0), (-1, -1), "TOP"),
                ("LEFTPADDING", (0, 0), (-1, -1), 0),
                ("RIGHTPADDING", (0, 0), (-1, -1), 0),
                ("TOPPADDING", (0, 0), (-1, -1), 0),
                ("BOTTOMPADDING", (0, 0), (-1, -1), 0),
            ])
        )

        row_table = Table(
            [[icon, text_table]],
            colWidths=[col_icon_w, col_text_w],
            style=TableStyle([
                ("VALIGN", (0, 0), (-1, -1), "TOP"),
                ("LEFTPADDING", (0, 0), (-1, -1), 0),
                ("RIGHTPADDING", (0, 0), (-1, -1), 0),
                ("TOPPADDING", (0, 0), (-1, -1), 2),
                ("BOTTOMPADDING", (0, 0), (-1, -1), 6),
            ])
        )

        # Optional: show confidence text lightly (remove if you truly want only the icon)
        # conf_line = Paragraph(f"confidence: {conf}", small)
        # entry = KeepTogether([row_table, conf_line, Spacer(1, 0.02 * inch)])

        entry = KeepTogether([row_table])
        story.append(entry)

    doc.build(
        story,
        onFirstPage=draw_header,
        onLaterPages=draw_header,
        canvasmaker=NumberedCanvas
    )


    return out_pdf

In [10]:

out_dir = Path("DataFrames")
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / f"references_verified.parquet"

df = pd.read_parquet(out_path)

In [11]:
# Example: df is your references dataframe already built above.
# It MUST have: ref_num, confidence, url; title is optional.

pdf_path = build_pdf_report(
    df,
    out_pdf="Kilbreths_Pig_Report.pdf",
    refnum_col="ref_id",
    confidence_col="verify_status",
    url_col="verify_evidence_url",
    title_col="title_final",
    evaluated_article="pHast_cam_DRAFT_revA.docx",
    about_sentence="This report summarizes automated verification results for the reference list associated with the manuscript under review.",
    logo_path="logo.png",
    yes_icon="yes.png",
    maybe_icon="maybe.png",
    no_icon="no.png",
)



pdf_path


'Kilbreths_Pig_Report.pdf'

In [12]:
df.columns.tolist()


['ref_id',
 'raw',
 'year',
 'doi',
 'urls',
 'title_guess',
 'authors',
 'title',
 'journal_or_source',
 'year_llm',
 'volume',
 'issue',
 'pages',
 'doi_llm',
 'url_llm',
 'ref_id_is_missing',
 'ref_id_is_duplicate',
 'title_final',
 'verify_status',
 'verify_score',
 'verify_best_source',
 'verify_best_match',
 'verify_evidence_url',
 'verify_notes']